# 🎯 Module 13 · 10 — Advanced Classical NLP

> **Stage:** Machine Learning → NLP  
> **Level:** Advanced  
> **Module:** 13 — NLP Basics / 10 — Advanced Classical NLP  
> **Date:** 3 February 2026  
> **Previous:** 09 — NLP Evaluation

---

This is the **final notebook** in Module 13. We combine everything we've learned into **advanced classical NLP** techniques:

- **Named Entity Recognition (NER)**
- **Regex** information extraction
- **Keyword extraction** (TF-IDF, TextRank)
- **Topic modeling** (LDA)
- **Text clustering** (K-Means)
- **Document similarity** (cosine)
- **Information Retrieval** / search
- **Extractive summarization**
- **Long documents** (chunking)
- **Multilingual / Roman Urdu / code-switching**
- **Unicode** handling
- **Domain-specific** NLP
- **Production** pipeline

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Extract entities with **NER**
- Use **regex** for structured information
- Extract **keywords** from documents
- Perform **LDA topic modeling**
- Cluster documents with **K-Means + TF-IDF**
- Compute **document similarity** with cosine
- Build a simple **search engine**
- Perform **extractive summarization**
- Handle **long documents** via chunking
- Reason about **multilingual**, **Roman Urdu**, and **code-switching**
- Handle **Unicode** correctly
- Reason about **domain-specific** NLP
- Design an **end-to-end classical NLP system**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What Is Advanced Classical NLP?

Using **traditional NLP techniques** to solve **more complex language problems**.

Examples:

```text
Named Entity Recognition
Topic Modeling
Text Clustering
Keyword Extraction
Information Extraction
Document Similarity
Information Retrieval
Search
Long-document processing
Multilingual text processing
Domain-specific NLP
```

The general idea remains:

```text
Text
 ↓
Preprocessing
 ↓
Representation
 ↓
Algorithm
 ↓
Useful Information
```

---
## 🏷️ 2. Named Entity Recognition (NER)

**NER** identifies important **entities** in text.

Example:

```text
"Apple opened a new office in London in 2025."
```

```text
Apple   → ORGANIZATION
London  → LOCATION
2025    → DATE
```

### Common entity types

| Entity | Example |
| --- | --- |
| PERSON | Elon Musk |
| ORGANIZATION | Microsoft |
| LOCATION | Islamabad |
| DATE | January 2026 |
| MONEY | $500 |
| PRODUCT | iPhone |

### Uses

- Document processing
- Search
- Resume analysis
- News analysis
- Customer support
- Information extraction

In [ ]:
# Try spaCy NER; fall back to a tiny regex-based demo
text = "Apple opened a new office in London in 2025. Ali joined Microsoft in Islamabad."

try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        raise ImportError('en_core_web_sm not installed')

    doc = nlp(text)
    print('Entities found by spaCy:')
    for ent in doc.ents:
        print(f'  {ent.text:<15} -> {ent.label_}')
except ImportError:
    print('spaCy / en_core_web_sm not available. Showing regex-based fallback:')
    # Very simple rules for demo purposes only
    org_pattern = r'\b(Apple|Microsoft)\b'
    loc_pattern = r'\b(London|Islamabad)\b'
    date_pattern = r'\b(20\d{2})\b'
    print('  ORGANIZATION:', re.findall(org_pattern, text))
    print('  LOCATION    :', re.findall(loc_pattern, text))
    print('  DATE        :', re.findall(date_pattern, text))

---
## 🔍 3. Regex & Rule-Based Extraction

Not every NLP problem needs ML. **Regex** is powerful for **structured patterns**.

```python
import re
re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', text)
```

### Great for

- Email addresses
- Phone numbers
- URLs
- Dates
- IDs / invoice numbers

### ⚠️ But regex has limits

It cannot understand **deeper language meaning** — e.g. `Apple` the company vs `apple` the fruit.

In [ ]:
# Regex extraction of emails, phones, and URLs
text = '''
Contact us at test@example.com or support@site.co.
Call +92-300-1234567 for help.
Visit https://example.com or www.demo.io for more.
'''

emails = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', text)
phones = re.findall(r'\+?\d[\d\-\s]{8,}\d', text)
urls   = re.findall(r'https?://\S+|www\.\S+', text)

print('Emails :', emails)
print('Phones :', phones)
print('URLs   :', urls)

---
## 🧾 4. Information Extraction

Turn **unstructured text** into **structured information**.

```text
"Ali joined Microsoft in Islamabad in 2025."
```

```text
Person       → Ali
Organization → Microsoft
Location     → Islamabad
Date         → 2025
```

Very useful for:

- Resume parsing
- Financial document analysis
- News analysis
- Customer support ticketing

---
## 🔑 5. Keyword Extraction

Identifies the most **important words or phrases** in a document.

Example:

```text
Document:
"Machine learning allows computers to learn patterns
from data and make predictions."
```

Possible keywords:

```text
machine learning
computers
patterns
data
predictions
```

### Classical approaches

- **TF-IDF**
- Frequency-based methods
- N-grams
- **TextRank**
- **RAKE**

---
## 📊 6. TF-IDF for Keyword Extraction

TF-IDF identifies terms that are **frequent in a document** but **rare across the corpus**.

Example corpus:

```text
Doc A: "machine learning models predict house prices"
Doc B: "machine learning models classify images"
```

Words like:

```text
house
prices
classify
images
```

become **most distinguishing** for each document.

In [ ]:
# Keyword extraction with TF-IDF
docs = [
    "machine learning models predict house prices",
    "machine learning models classify images",
    "deep learning neural networks classify images",
]

vec = TfidfVectorizer(stop_words='english')
X = vec.fit_transform(docs)
terms = vec.get_feature_names_out()

for i, doc in enumerate(docs):
    row = X[i].toarray().ravel()
    top_idx = row.argsort()[::-1][:5]
    print(f'Doc {i+1}: {doc}')
    print('  Top keywords:')
    for idx in top_idx:
        if row[idx] > 0:
            print(f'    {terms[idx]:<12} {row[idx]:.3f}')
    print()

---
## 🔗 7. TextRank

**TextRank** is a **graph-based** algorithm inspired by PageRank.

```text
Text
 ↓
Words / Sentences
 ↓
Graph
 ↓
Importance scores
 ↓
Important keywords / sentences
```

Used for:

- **Keyword extraction**
- **Extractive summarization**

Unlike simple frequency counting, TextRank considers **relationships** between elements.

---
## 🗂️ 8. Topic Modeling

Discovers **hidden topics** in a collection of documents — **unsupervised**.

Suppose we have thousands of news articles. The model may discover:

```text
Topic 1 → Technology
Topic 2 → Sports
Topic 3 → Politics
Topic 4 → Business
```

No labels are provided to the algorithm.

---
## 📚 9. Latent Dirichlet Allocation (LDA)

**LDA** (Latent Dirichlet Allocation — **not** Linear Discriminant Analysis) assumes documents are **mixtures of topics**.

Example:

```text
Document:
"AI companies are developing new machine learning models."

Topic distribution:
Technology → 80%
Business   → 15%
Other      →  5%
```

LDA represents:

```text
Document
   ↓
Topic Distribution
   ↓
Word Distribution
```

In [ ]:
# Tiny LDA example
docs = [
    "football match player goal team",
    "football team championship goal",
    "stock market company investment revenue",
    "company revenue financial market stock",
    "football league player match goal",
    "investment market stock revenue company",
]

vec = CountVectorizer()
X = vec.fit_transform(docs)

lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X)

terms = vec.get_feature_names_out()
print('Topics discovered by LDA:')
for topic_idx, topic in enumerate(lda.components_):
    top_terms = [terms[i] for i in topic.argsort()[::-1][:5]]
    print(f'  Topic {topic_idx + 1}: ' + ', '.join(top_terms))

---
## ⚠️ 10. Limitations of Topic Modeling

- Difficult topic interpretation
- Similar topics
- Poor on **short documents**
- Sensitive to preprocessing
- Choosing the **number of topics** is subjective
- Generic words can dominate

> Always **inspect and interpret** topics — don't accept them blindly.

---
## 🧩 11. Text Clustering

Groups **similar documents** without predefined labels.

```text
Documents
    ↓
TF-IDF
    ↓
K-Means
    ↓
Clusters
```

Possible result:

```text
Cluster 1 → Sports
Cluster 2 → Technology
Cluster 3 → Business
```

### Uses

- Document organization
- Customer segmentation
- News grouping
- Support ticket grouping

In [ ]:
# K-Means clustering on TF-IDF text vectors
docs = [
    "football match player goal team",
    "football team championship goal",
    "football league player match",
    "machine learning models predict house prices",
    "machine learning models classify images",
    "deep learning neural networks classify images",
    "stock market company investment revenue",
    "company revenue financial market stock",
]

vec = TfidfVectorizer(stop_words='english')
X = vec.fit_transform(docs)

km = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels = km.fit_predict(X)

print('Cluster assignment:')
for doc, lab in zip(docs, labels):
    print(f'  [cluster {lab}] {doc}')

---
## 📐 12. Document Similarity

How **similar** are two pieces of text?

```text
Document A → Vector
Document B → Vector
       ↓
Cosine Similarity
       ↓
Similarity Score
```

Example:

```text
A: "machine learning tutorial"
B: "machine learning course"
```

→ **high similarity**

---
## 📏 13. Cosine Similarity

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

Range:

```text
-1 → opposite direction
 0 → no similarity
+1 → same direction
```

```python
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(document_vectors)
```

In [ ]:
# Pairwise document similarity
docs = [
    "machine learning tutorial",
    "machine learning course",
    "football match highlights",
]
vec = TfidfVectorizer()
X = vec.fit_transform(docs)
sim = cosine_similarity(X)

print('Similarity matrix:')
print(pd.DataFrame(sim, index=docs, columns=docs).round(3))

---
## 🔎 14. Information Retrieval (IR)

Finding **relevant information** from a collection.

```text
User Query
     ↓
Search System
     ↓
Retrieve Documents
     ↓
Rank Documents
     ↓
Relevant Results
```

A classical search system uses:

```text
TF-IDF + Cosine Similarity
```

---
## 🧪 15. Classical Search Pipeline

```text
Documents
   ↓
Clean Text
   ↓
TF-IDF
   ↓
Store Vectors

User Query
   ↓
Same TF-IDF
   ↓
Query Vector
   ↓
Cosine Similarity
   ↓
Rank Documents
   ↓
Return Results
```

In [ ]:
# Tiny search engine with TF-IDF + cosine similarity
docs = [
    "python machine learning tutorial",
    "java web development guide",
    "machine learning with python",
    "deep learning with neural networks",
    "football match highlights today",
]
vec = TfidfVectorizer()
X = vec.fit_transform(docs)

def search(query, top_k=3):
    q = vec.transform([query])
    sims = cosine_similarity(q, X).ravel()
    ranked = sorted(enumerate(sims), key=lambda x: -x[1])
    return [(docs[i], round(s, 4)) for i, s in ranked[:top_k]]

for q in ["python machine learning", "football"]:
    print(f'Query: {q!r}')
    for doc, score in search(q):
        print(f'  {score:>6.4f}  {doc}')
    print()

---
## 🆚 16. Keyword vs Semantic Search

### Keyword search
```text
"car repair" → matches "car repair service"
```
✅ Fast, interpretable, strong baseline.

### Semantic search
```text
"car repair" → should also match "vehicle maintenance center"
```
✅ Handles synonyms and paraphrase — needs **embeddings**.

Modern search systems often combine both.

---
## 📚 17. Long Documents

Examples:

```text
Books
Legal documents
Research papers
Company reports
PDFs
Contracts
```

### Strategy: chunking

```text
Long Document
      ↓
Chunks
      ↓
Process Each Chunk
      ↓
Combine Results
```

This idea becomes **very important** later in **RAG systems**.

In [ ]:
# Simple chunking by word windows
long_text = " ".join([f"sentence_{i}" for i in range(1, 21)])

def chunk(text, size=5, overlap=0):
    words = text.split()
    chunks = []
    step = size - overlap
    for i in range(0, len(words), step):
        chunk = words[i:i + size]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks

chunks = chunk(long_text, size=5, overlap=1)
for i, c in enumerate(chunks, 1):
    print(f'Chunk {i}: {c}')

---
## ✂️ 18. Extractive Summarization

Selects important **sentences** from an existing document.

```text
Original Document
       ↓
Sentence Scoring
       ↓
Important Sentences
       ↓
Summary
```

### Classical methods

- TF-IDF
- TextRank
- Sentence frequency
- Position
- Similarity

In [ ]:
# Extractive summarization: score sentences by TF-IDF word weights
doc = (
    "Machine learning is transforming industries. "
    "It allows computers to learn patterns from data. "
    "Deep learning uses neural networks with many layers. "
    "Transformers changed natural language processing. "
    "Football is a popular sport. "
    "Machine learning models can make predictions. "
)

sentences = [s.strip() for s in doc.split('.') if s.strip()]

# Compute TF-IDF over sentences as "documents"
vec = TfidfVectorizer(stop_words='english')
X = vec.fit_transform(sentences)

# Score each sentence by sum of TF-IDF weights
scores = np.asarray(X.sum(axis=1)).ravel()
order = scores.argsort()[::-1]

print('Top 3 sentences (extractive summary):')
for i in order[:3]:
    print(f'  [score={scores[i]:.2f}] {sentences[i]}.')

---
## 🆚 19. Extractive vs Abstractive Summarization

| | Extractive | Abstractive |
| --- | --- | --- |
| Method | Select existing sentences | Generate new text |
| Risk | May be redundant | May hallucinate |
| Where used | Classical NLP | Deep Learning / Transformers |

Abstractive summarization belongs to **modern NLP** — its coverage is in the **Generative AI** module.

---
## 🌍 20. Multilingual NLP

Different languages have different:

- Writing systems
- Word structures
- Grammar
- Tokenization rules
- Stop words
- Stemming requirements

Examples:

```text
English
Urdu
Arabic
Chinese
German
```

> **Do not** reuse the same preprocessing for every language.

---
## 🗣️ 21. Urdu & Roman Urdu NLP

**Roman Urdu** introduces extra challenges:

```text
"ye product bohat acha hai"
"ye product boht acha hy"
"ya product bohat acha ha"
```

Possible issues:

- Spelling variation
- English + Urdu mixing
- Informal grammar
- Romanization
- Many equivalent spellings

May need:
- Custom normalization
- Character n-grams
- Domain-specific training data

---
## 🧬 22. Code-Switching

Multiple languages in the same sentence:

```text
"ye phone bohat acha hai but battery weak hai"
```

### Possible approaches

- **Language detection**
- Normalization
- Character n-grams
- Multilingual datasets
- Language-specific preprocessing

---
## 🌐 23. Unicode Handling

Text is not just ASCII.

```text
English → Hello
Urdu    → ہیلو
Arabic  → مرحبا
Emoji   → 😊
```

Handle:

- Encoding
- Normalization
- Special characters
- Emoji
- Different scripts

> Incorrect Unicode handling can corrupt text **before** it reaches the model.

In [ ]:
# Demonstrate Unicode handling
text = "Hello ہیلو مرحبا 😊"

print('Text           :', text)
print('Length         :', len(text))
print('Codepoints     :', [f'U+{ord(c):04X}' for c in text if ord(c) > 0x2000][:6], '...')

# Normalize with unicodedata
import unicodedata
normalized = unicodedata.normalize('NFKC', text)
print('NFKC-normalized:', normalized)

---
## 🎯 24. Domain-Specific NLP

A model trained on general text often performs **poorly** in specialized domains.

```text
General NLP    → General vocabulary
Medical NLP    → Medical terminology
Legal NLP      → Legal terminology
Financial NLP  → Financial terminology
```

> Domain-specific **data** and **vocabulary** can dramatically improve performance.

---
## 🛠️ 25. Feature Engineering

### Word features
```text
word frequency
TF-IDF
```

### N-grams
```text
unigrams / bigrams / trigrams
```

### Character features
```text
character n-grams
```

### Document features
```text
document length
word count
sentence count
```

### Special features
```text
emoji count
URL count
hashtag count
uppercase count
```

Combine them depending on the task.

---
## 🔤 26. Word vs Character Features

```text
"amazinggg"
```

- Word-based → treats as a **completely new** word
- Character n-grams → captures `amaz`, `amazi`, `zing`, `ingg`

### Character features help with

- Misspellings
- Slang
- Usernames
- Social media text
- Morphological variations
- Roman Urdu

---
## 🎛️ 27. Controlling the Vocabulary

Large text datasets can produce **millions of features**. That increases memory, training time, noise, and overfitting risk.

```python
TfidfVectorizer(min_df=2, max_df=0.95, max_features=50000)
```

- `min_df` → removes extremely rare terms
- `max_df` → removes terms appearing in too many docs
- `max_features` → limits vocabulary size

---
## 0️⃣ 28. Binary Features

Sometimes **presence** matters more than **frequency**.

```text
word frequency = 7   →    present = 1
word frequency = 0   →    absent  = 0
```

```python
CountVectorizer(binary=True)
```

---
## 🏗️ 29. Advanced NLP Project Architecture

```text
NLP Project
│
├── data/
│   ├── raw/
│   └── processed/
│
├── notebooks/
│   ├── exploration.ipynb
│   └── modeling.ipynb
│
├── src/
│   ├── preprocessing.py
│   ├── features.py
│   ├── training.py
│   └── prediction.py
│
├── models/
│   └── model.pkl
│
├── app/
│   └── main.py
│
├── requirements.txt
└── README.md
```

Separate **experimentation** from **reusable application code**.

---
## 🔁 30. End-to-End Classical NLP System

```text
                    Dataset
                       ↓
               Data Cleaning
                       ↓
              Train/Test Split
                       ↓
             Text Preprocessing
                       ↓
               TF-IDF / BoW
                       ↓
                ML Algorithm
                       ↓
                 Evaluation
                       ↓
                Error Analysis
                       ↓
               Hyperparameter Tuning
                       ↓
                Final Pipeline
                       ↓
                  Save Model
                       ↓
                   FastAPI
                       ↓
                    Docker
                       ↓
                 Production
```

---
## 🚀 31. Classical NLP Production Pipeline

```text
User
 ↓
API
 ↓
Input Validation
 ↓
Preprocessing
 ↓
Feature Extraction
 ↓
Model
 ↓
Prediction
 ↓
Post-processing
 ↓
Response
```

```json
POST /predict
{"text": "I really love this product"}

→ {"sentiment": "positive"}
```

---
## 🌍 32. Common Classical NLP Applications

### Document Classification
```text
Document → TF-IDF → Classifier → Category
```

### Spam Detection
```text
Email → Features → Classifier → Spam / Not Spam
```

### Sentiment Analysis
```text
Review → TF-IDF → Classifier → Positive / Negative
```

### Document Similarity
```text
Document A ↔ Document B
```

### Search
```text
Query → TF-IDF → Similarity → Ranked Documents
```

### Topic Discovery
```text
Documents → Topic Modeling → Discovered Topics
```

---
## ⚠️ 33. Classical NLP Limitations

Classical NLP struggles with:

- Deep context
- Long-range dependencies
- Sarcasm
- Ambiguous language
- Complex reasoning
- Context-dependent meanings
- Large-scale multilingual understanding
- Generating natural language

Example:

```text
"I saw the man with the telescope."
```

Multiple interpretations — feature-based methods struggle.

---
## 🧭 34. Classical NLP → Deep Learning

```text
Classical NLP
     ↓
BoW
     ↓
TF-IDF
     ↓
Static Word Embeddings
     ↓
Neural Networks
     ↓
RNN
     ↓
LSTM / GRU
     ↓
Attention
     ↓
Transformers
     ↓
BERT / GPT
     ↓
LLMs
     ↓
RAG
     ↓
Agents
```

> Classical NLP is the **foundation** that makes modern NLP understandable.

---
## 🎓 35. Practical Learning Projects

### 1. News Topic Discovery
```text
News Articles → TF-IDF → Clustering / LDA → Topics
```

### 2. Document Search Engine
```text
Documents → TF-IDF → User Query → Cosine → Ranked Results
```

### 3. Resume Classification
```text
Resume → TF-IDF → Classifier → Job Category
```

### 4. Keyword Extractor
```text
Document → TF-IDF / TextRank → Important Keywords
```

### 5. Document Similarity Tool
```text
Document A → Vector → Cosine Similarity ← Vector ← Document B
```

---
## 📋 36. Recommended Classical NLP Workflow

```text
1. Understand the task
        ↓
2. Inspect the dataset
        ↓
3. Clean the text
        ↓
4. Build a simple baseline
        ↓
5. Try TF-IDF
        ↓
6. Train a simple model
        ↓
7. Evaluate
        ↓
8. Perform error analysis
        ↓
9. Engineer useful features
        ↓
10. Tune the model
        ↓
11. Compare approaches
        ↓
12. Save the pipeline
        ↓
13. Deploy if required
```

> Do **not** start with the most complicated technique. Start simple, then improve based on evidence.

---
## 🧰 37. Important Tools

| Tool | Purpose |
| --- | --- |
| **NLTK** | NLP fundamentals |
| **spaCy** | NLP processing & NER |
| **scikit-learn** | ML & text features |
| **Gensim** | Topic modeling & embeddings |
| **Pandas** | Data processing |
| **NumPy** | Numerical operations |
| **Matplotlib** | Visualization |
| **Regex** | Pattern extraction |

---
## 📄 38. Classical NLP Cheat Sheet

| Problem | Classical Approach |
| --- | --- |
| Text classification | TF-IDF + ML |
| Sentiment | TF-IDF + classifier |
| Spam detection | TF-IDF + classifier |
| Similarity | TF-IDF + cosine |
| Search | TF-IDF + ranking |
| Topic discovery | LDA |
| Document clustering | K-Means |
| Keyword extraction | TF-IDF / TextRank |
| Pattern extraction | Regex |
| Entity extraction | NER |
| Summarization | TextRank |
| Misspellings | Edit distance |
| Social media text | Character n-grams |
| Long documents | Chunking |
| Multilingual text | Language-aware preprocessing |

---
## 🧠 39. Final Mental Model

```text
                 RAW TEXT
                    ↓
             PREPROCESSING
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
      WORD FEATURES      CHARACTER FEATURES
          ↓                   ↓
       BoW / TF-IDF       N-Grams
          ↓                   ↓
          └─────────┬─────────┘
                    ↓
              ML / NLP METHOD
                    ↓
       ┌────────────┼────────────┐
       ↓            ↓            ↓
 Classification   Clustering   Search
       ↓            ↓            ↓
 Sentiment        Topics      Similarity
       ↓
    Evaluation
       ↓
 Error Analysis
       ↓
   Improvement
       ↓
   Deployment
```

### 🧭 Golden Rule

> **Classical NLP turns unstructured language into useful numerical features and then uses algorithms to classify, compare, organize, search, or extract information from text.**

---
## 🏋️ 40. Hands-On Exercises

### Exercise 1 — NER
Extract all entities from:
```text
"Microsoft opened a new office in Dubai in 2024."
```
Categorize as PERSON / ORGANIZATION / LOCATION / DATE.

### Exercise 2 — Regex
Write regex to extract:
- Emails
- Phone numbers
- URLs
from a paragraph of text.

### Exercise 3 — Keywords
Given a document about ML, extract the top-5 keywords using TF-IDF.

### Exercise 4 — Topic modeling
Run LDA with `n_components=3` on a small corpus of your own. Interpret each topic in 1 sentence.

### Exercise 5 — Clustering
Cluster 20 short news headlines into 3 groups using TF-IDF + K-Means. Interpret each cluster.

### Exercise 6 — Search engine
Build a tiny search engine over 10 documents. Test 3 queries.

### Exercise 7 — Summarization
Implement a 3-sentence extractive summarizer using TF-IDF sentence scoring.

### Exercise 8 — Multilingual
Take a Roman Urdu sentence and try common preprocessing steps. What breaks?

---
## 🎤 41. Interview Questions

1. **What is NER?**  
   Named Entity Recognition — extracting entities like PERSON, ORG, LOCATION, DATE.

2. **When is regex enough?**  
   For predictable, structured patterns — emails, phones, URLs, IDs.

3. **What is Information Extraction?**  
   Converting unstructured text into structured data.

4. **How does TF-IDF help with keywords?**  
   It scores terms that are frequent in a doc but rare across the corpus.

5. **What is TextRank?**  
   A graph-based algorithm for keywords and extractive summarization.

6. **What is LDA (topic modeling)?**  
   A probabilistic model that assumes documents are mixtures of topics.

7. **LDA topic modeling vs Linear Discriminant Analysis?**  
   Different algorithms with the same acronym — one unsupervised, one supervised.

8. **What is K-Means for text?**  
   Clustering TF-IDF document vectors into groups.

9. **What is cosine similarity?**  
   A measure of the angle between two vectors — the standard for text similarity.

10. **What is Information Retrieval?**  
    Retrieving relevant documents from a collection in response to a query.

11. **Keyword vs semantic search?**  
    Keyword matches exact terms; semantic matches meanings (needs embeddings).

12. **What is extractive summarization?**  
    Selecting the most important sentences from a document.

13. **Why chunk long documents?**  
    To process content that doesn't fit in a single pass — a key idea in RAG.

14. **Why is Roman Urdu hard?**  
    Many spelling variants, code-switching, informal grammar, no standard normalization.

15. **Why handle Unicode carefully?**  
    Text may include emojis, Arabic, Urdu, etc. Bad handling corrupts input.

16. **Why is domain-specific NLP needed?**  
    Terminology and meaning vary by domain (medical, legal, financial).

17. **When do character n-grams help?**  
    Misspellings, slang, short text, morphologically rich languages.

18. **Why limit vocabulary?**  
    Reduces memory, training time, noise, and overfitting.

19. **What is binary bag-of-words?**  
    Records word presence (1) rather than count.

20. **Main limitation of classical NLP?**  
    Limited context and semantics — motivated deep learning and Transformers.

---
## 🏁 42. Key Takeaways

### You now understand

- **NER** & entity types
- **Regex** information extraction
- **Keyword extraction** (TF-IDF, TextRank)
- **Topic modeling** with **LDA**
- **Text clustering** with K-Means
- **Document similarity** with cosine
- **Information Retrieval** & search
- **Extractive summarization**
- **Long-document** chunking
- **Multilingual** & **Roman Urdu** & **code-switching**
- **Unicode** handling
- **Domain-specific** NLP
- **Word vs character** features
- **Vocabulary control**
- **Binary features**
- **Project architecture**
- **End-to-end pipelines**
- **Production** workflow
- **Classical limitations**
- **Classical → Deep Learning** progression

### The big idea

> **Classical NLP turns unstructured language into useful numerical features and then uses algorithms to classify, compare, organize, search, or extract information from text.**

---

## 🎯 Module 13 — Complete!

You've finished the entire classical NLP foundation:

```text
01 NLP Fundamentals
        ↓
02 Text Preprocessing
        ↓
03 Text Representation
        ↓
04 Bag of Words
        ↓
05 TF-IDF
        ↓
06 Word Embeddings
        ↓
07 Text Classification
        ↓
08 Sentiment Analysis
        ↓
09 NLP Evaluation
        ↓
10 Advanced Classical NLP ✅
```

---

### 🔗 What's Next?

**Module 14 — ML Pipelines**

We'll build production-ready **scikit-learn Pipelines**, **ColumnTransformers**, and **custom transformers** — the plumbing that makes all of this deployable.